# 01. QC & トリミング        RNA-seq FASTQデータの品質管理とアダプター除去を行います。**カーネル**: RNA-seq (Python) / rnaseq_env  **必要ツール**: FastQC, MultiQC, Trim Galore  **入力**: `raw_data/` 内のFASTQファイル (Paired-end)  **出力**: `trimmed/` にトリミング済みFASTQ, `qc_reports/` にQCレポート

## 設定

In [ ]:
import os, glob, subprocess, time

# === 設定 ===
RAW_DATA_DIR = "raw_data"
TRIMMED_DIR = "trimmed"
QC_DIR = "qc_reports"
QC_PRETRIM_DIR = os.path.join(QC_DIR, "pretrim")
QC_POSTTRIM_DIR = os.path.join(QC_DIR, "posttrim")
THREADS = 8

# ディレクトリ作成
for d in [TRIMMED_DIR, QC_PRETRIM_DIR, QC_POSTTRIM_DIR]:
    os.makedirs(d, exist_ok=True)

print("設定完了")

## ツール確認

In [ ]:
# 必要ツールの存在確認
for tool in ["fastqc", "multiqc", "trim_galore"]:
    ret = os.system(f"which {{tool}} > /dev/null 2>&1")
    status = "OK" if ret == 0 else "NOT FOUND"
    print(f"  {{tool}}: {{status}}")

## FASTQファイルの検出

In [ ]:
# Paired-end FASTQファイルを検出
r1_files = sorted(glob.glob(os.path.join(RAW_DATA_DIR, "*_R1*.fastq.gz")))
r2_files = sorted(glob.glob(os.path.join(RAW_DATA_DIR, "*_R2*.fastq.gz")))

print(f"検出されたペア数: {{len(r1_files)}}")
for r1, r2 in zip(r1_files, r2_files):
    print(f"  R1: {{os.path.basename(r1)}}")
    print(f"  R2: {{os.path.basename(r2)}}")
    print()

assert len(r1_files) == len(r2_files), "R1とR2のファイル数が一致しません！"
assert len(r1_files) > 0, "FASTQファイルが見つかりません。raw_data/ を確認してください。"

## Step 1: トリミング前 QC (FastQC)

In [ ]:
%%time
all_fastqs = " ".join(r1_files + r2_files)
cmd = f"fastqc -t {THREADS} -o {QC_PRETRIM_DIR} {all_fastqs}"
print(f"実行: {cmd}")
os.system(cmd)
print("FastQC (トリミング前) 完了")

## Step 2: トリミング (Trim Galore)

In [ ]:
%%time
for r1, r2 in zip(r1_files, r2_files):
    cmd = (
        f"trim_galore --paired --quality 20 --length 36 "
        f"--cores {THREADS} "
        f"-o {TRIMMED_DIR} "
        f"{r1} {r2}"
    )
    print(f"処理中: {os.path.basename(r1)}")
    os.system(cmd)

print("\nTrim Galore 完了")
print(f"トリミング済みファイル: {TRIMMED_DIR}/")

## Step 3: トリミング後 QC (FastQC)

In [ ]:
%%time
trimmed_fastqs = " ".join(sorted(glob.glob(os.path.join(TRIMMED_DIR, "*_val_*.fq.gz"))))
cmd = f"fastqc -t {THREADS} -o {QC_POSTTRIM_DIR} {trimmed_fastqs}"
print(f"実行: {cmd}")
os.system(cmd)
print("FastQC (トリミング後) 完了")

## Step 4: MultiQC レポート統合

In [ ]:
%%time
cmd = f"multiqc {QC_DIR} -o {QC_DIR} -f --title 'RNA-seq QC Report'"
os.system(cmd)
print(f"MultiQCレポート: {QC_DIR}/multiqc_report.html")

## サマリー- トリミング前後のQCレポートを `qc_reports/multiqc_report.html` で確認してください- 品質に問題がなければ次のノートブック `02_Mapping_and_Counting.ipynb` に進みます### チェックポイント- Per base quality scores が Q20 以上であること- Adapter content が除去されていること  - 極端にリード数が少ないサンプルがないこと